# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [5]:
# Write your code below.
%load_ext dotenv
%dotenv 


In [32]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [33]:
import os
from glob import glob

# Write your code below.

# Hardcoded absolute path to your project data root
price_data_dir = "/Users/kashfa/DSI/production/05_src/data/prices/"

# Verify path
print("Hardcoded PRICE_DATA path:", price_data_dir)

# Recursive glob to search all parquet files inside subfolders
parquet_files = glob(os.path.join(price_data_dir, '**/*.parquet'), recursive=True)
dd_px = dd.read_parquet(parquet_files).set_index("ticker")
parquet_files



Hardcoded PRICE_DATA path: /Users/kashfa/DSI/production/05_src/data/prices/


['/Users/kashfa/DSI/production/05_src/data/prices/BKTI/BKTI_2012/part.0.parquet',
 '/Users/kashfa/DSI/production/05_src/data/prices/BKTI/BKTI_2012/part.1.parquet',
 '/Users/kashfa/DSI/production/05_src/data/prices/BKTI/BKTI_2015/part.0.parquet',
 '/Users/kashfa/DSI/production/05_src/data/prices/BKTI/BKTI_2015/part.1.parquet',
 '/Users/kashfa/DSI/production/05_src/data/prices/BKTI/BKTI_2014/part.0.parquet',
 '/Users/kashfa/DSI/production/05_src/data/prices/BKTI/BKTI_2014/part.1.parquet',
 '/Users/kashfa/DSI/production/05_src/data/prices/BKTI/BKTI_2013/part.0.parquet',
 '/Users/kashfa/DSI/production/05_src/data/prices/BKTI/BKTI_2013/part.1.parquet',
 '/Users/kashfa/DSI/production/05_src/data/prices/BKTI/BKTI_1980/part.0.parquet',
 '/Users/kashfa/DSI/production/05_src/data/prices/BKTI/BKTI_1980/part.1.parquet',
 '/Users/kashfa/DSI/production/05_src/data/prices/BKTI/BKTI_1987/part.0.parquet',
 '/Users/kashfa/DSI/production/05_src/data/prices/BKTI/BKTI_1987/part.1.parquet',
 '/Users/kashfa/

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [34]:


# Read parquet files into one large Dask dataframe
ddf = dd.read_parquet(parquet_files)

# Normalize column names
ddf = ddf.rename(columns={
    'ticker': 'Ticker',
    'Adj Close': 'Adj_Close'
})

# Sort before repartitioning
ddf = ddf.sort_values(by=["Ticker", "Date"])

# Repartition by Ticker fully to consolidate rows together
ddf = ddf.set_index("Ticker", shuffle="tasks")

In [35]:
def feature_engineering(df_partition):
    df_partition = df_partition.sort_values(by=["Date"])
    df_partition["Close_lag_1"] = df_partition["Close"].shift(1)
    df_partition["Adj_Close_lag_1"] = df_partition["Adj_Close"].shift(1)
    df_partition["returns"] = (df_partition["Close"] / df_partition["Close_lag_1"]) - 1
    df_partition["hi_lo_range"] = df_partition["High"] - df_partition["Low"]
    df_partition = df_partition.dropna(subset=["Close_lag_1", "Adj_Close_lag_1"])
    return df_partition

meta = {
    'Date': 'datetime64[ns]',
    'Open': 'float64',
    'High': 'float64',
    'Low': 'float64',
    'Close': 'float64',
    'Adj_Close': 'float64',
    'Volume': 'float64',
    'source': 'object',
    'Year': 'int64',
    'Close_lag_1': 'float64',
    'Adj_Close_lag_1': 'float64',
    'returns': 'float64',
    'hi_lo_range': 'float64'
}

dd_feat = ddf.map_partitions(feature_engineering, meta=meta)



In [36]:
dd_feat.compute()

,Date,Open,High,Low,Close,Adj_Close,Volume,source,Year,Close_lag_1,Adj_Close_lag_1,returns,hi_lo_range
Ticker,,,,,,,,,,,,,
A,1999-11-19,30.713520,30.758226,28.478184,28.880543,24.838577,15234100.0,A.csv,1999,31.473534,27.068665,-0.082386,2.280043
A,1999-11-22,29.551144,31.473534,28.657009,31.473534,27.068665,6577800.0,A.csv,1999,28.880543,24.838577,0.089783,2.816525
A,1999-11-23,30.400572,31.205294,28.612303,28.612303,24.607880,5975600.0,A.csv,1999,31.473534,27.068665,-0.090909,2.592991
A,1999-11-24,28.701717,29.998211,28.612303,29.372318,25.261524,4843200.0,A.csv,1999,28.612303,24.607880,0.026563,1.385908
A,1999-11-26,29.238197,29.685265,29.148785,29.461731,25.338428,1729400.0,A.csv,1999,29.372318,25.261524,0.003044,0.536480
...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZEUS,2020-03-26,9.610000,9.940000,9.260000,9.590000,9.590000,60500.0,ZEUS.csv,2020,9.670000,9.670000,-0.008273,0.679999
ZEUS,2020-03-27,9.330000,9.330000,8.700000,8.700000,8.700000,52900.0,ZEUS.csv,2020,9.590000,9.590000,-0.092805,0.630000
ZEUS,2020-03-30,8.810000,9.760000,8.700000,9.680000,9.680000,73700.0,ZEUS.csv,2020,8.700000,8.700000,0.112644,1.060000


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [37]:
# Write your code below.

pdf = dd_feat.compute()
pdf = pdf.sort_values(by=["Ticker", "Date"])

pdf["returns_ma10"] = pdf.groupby("Ticker")["returns"].rolling(10).mean().reset_index(0, drop=True)

# View the final processed data
pdf.head()

,Date,Open,High,Low,Close,Adj_Close,Volume,source,Year,Close_lag_1,Adj_Close_lag_1,returns,hi_lo_range,returns_ma10
Ticker,,,,,,,,,,,,,,
A,1999-11-19,30.713520,30.758226,28.478184,28.880543,24.838577,15234100.0,A.csv,1999,31.473534,27.068665,-0.082386,2.280043,NaN
A,1999-11-22,29.551144,31.473534,28.657009,31.473534,27.068665,6577800.0,A.csv,1999,28.880543,24.838577,0.089783,2.816525,NaN
A,1999-11-23,30.400572,31.205294,28.612303,28.612303,24.607880,5975600.0,A.csv,1999,31.473534,27.068665,-0.090909,2.592991,NaN
A,1999-11-24,28.701717,29.998211,28.612303,29.372318,25.261524,4843200.0,A.csv,1999,28.612303,24.607880,0.026563,1.385908,NaN
A,1999-11-26,29.238197,29.685265,29.148785,29.461731,25.338428,1729400.0,A.csv,1999,29.372318,25.261524,0.003044,0.536480,NaN


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
 Yes, because for rolling calculations like .rolling(10), pandas needs full access to the entire group's data to work properly. Since Dask handles data in smaller pieces(partitions), it's harder to do rolling calculations directly in Dask without extra steps.Converting to pandas made it easier because pandas works on the whole dataset at once.

+ Would it have been better to do it in Dask? Why?
Technically yes, but working directly in Dask for rolling averages is more complicated and requires advanced partition handling. Since my dataset was small enough to fit into memory after feature engineering, using pandas here was simpler, more straightforward, and helped avoid complicated Dask configurations.

(1 pt)

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.